# 03 – Polars

> Runs against the catalog from `CATALOG` (Lakekeeper, Polaris, RustFS or Unity; default Lakekeeper). What a catalog cannot do is shown by the cell instead of aborting.

Polars uses PyIceberg for the catalog. **Writing** goes through PyIceberg (`write_iceberg`),
**reading** through Polars' own Rust reader (`scan_iceberg`). Its own reader does not take over
the table's S3 configuration automatically – we pass it on as `storage_options`.

> Limitations: no remote signing when reading; no types `uuid`, `fixed`; `time` is ns-precise in Polars.

In [ ]:
from iceberg_lab.config import PROFILE, REMOTE_SIGNING, SETTINGS, WAREHOUSE_SIGNING, WAREHOUSE_STS, WAREHOUSES
from iceberg_lab.notebook import BUCKET, TRINO_SIGNING, TRINO_STS, catalog_limit, drop_table, header

CATALOG_URI = SETTINGS.catalog_uri
S3_ENDPOINT = SETTINGS.s3_endpoint
header()

In [ ]:
import polars as pl
from iceberg_lab.engines import pyiceberg_catalog

from iceberg_lab.engines import polars_storage_options

catalog = pyiceberg_catalog(WAREHOUSE_STS)
catalog.create_namespace_if_not_exists("nb_polars")
if catalog.table_exists("nb_polars.trips"):
    drop_table(catalog, "nb_polars.trips")

## Create a DataFrame and write it as an Iceberg table

In [ ]:
trips = pl.DataFrame({
    "trip_id": range(1, 10_001),
    "zone": [f"Z{i % 7}" for i in range(10_000)],
    "distance_km": [round((i % 97) * 0.37, 2) for i in range(10_000)],
    "fare": [round(3.5 + (i % 97) * 0.9, 2) for i in range(10_000)],
})
table = catalog.create_table("nb_polars.trips", schema=trips.to_arrow().schema)
trips.write_iceberg(table, mode="append")
table.current_snapshot().summary

## Lazy scan with pushdown

In [ ]:
lf = pl.scan_iceberg(table, storage_options=polars_storage_options(table.io.properties))
result = (
    lf.filter(pl.col("distance_km") > 10)
      .group_by("zone")
      .agg(pl.len().alias("trips"), pl.col("fare").mean().round(2).alias("avg_fare"))
      .sort("zone")
)
print(result.explain())
result.collect()

## Overwrite and time travel

In [ ]:
first = table.current_snapshot().snapshot_id
trips.filter(pl.col("zone") == "Z0").write_iceberg(table, mode="overwrite")

now = pl.scan_iceberg(table.refresh(), storage_options=polars_storage_options(table.io.properties)).select(pl.len()).collect()
then = pl.scan_iceberg(table, snapshot_id=first, storage_options=polars_storage_options(table.io.properties)).select(pl.len()).collect()
print("now:", now.item(), " before:", then.item())